[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-USERNAME/AI-in-healthcare-book/blob/main/notebooks/chapter_04/notebook_4_5_data_leakage.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---


# Notebook 4.5: Data Leakage Detection and Prevention

**Chapter 4: Data Preparation**

## Learning Objectives

By the end of this notebook, you will be able to:
1. Identify common types of data leakage in healthcare ML
2. Detect target leakage and temporal leakage
3. Build proper preprocessing pipelines to prevent leakage
4. Quantify the impact of leakage on model performance
5. Design leakage-free workflows for clinical deployment

## Clinical Context

**Data leakage is the #1 reason models fail in deployment**:
- **Target leakage**: Using post-outcome information (e.g., post-sepsis labs to predict sepsis)
- **Temporal leakage**: Using future information (e.g., tomorrow's vital signs to predict today's outcome)
- **Train-test leakage**: Statistics from test set leak into training (e.g., scaling before splitting)

**Consequences**:
- Inflated performance estimates (AUROC 0.95+ with leakage, 0.75 without)
- Models that appear to work in development but fail completely in production
- Wasted resources, delayed deployments, loss of clinician trust

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

print("✓ Libraries imported")

## 1. Type 1: Target Leakage

**Definition**: Using information in features that is caused by or derived from the target variable.

**Example**: Predicting sepsis using labs ordered AFTER sepsis onset

In [ ]:
def generate_sepsis_data_with_leakage(n_patients=1000):
    """
    Generate ICU data with target leakage.
    """
    # Patient baseline
    age = np.random.normal(65, 15, n_patients)
    age = np.clip(age, 18, 100)

    comorbidity = np.random.choice([0, 1], n_patients, p=[0.7, 0.3])

    # Baseline vitals (before sepsis)
    baseline_hr = np.random.normal(80, 10, n_patients)
    baseline_wbc = np.random.normal(8, 2, n_patients)

    # Outcome: sepsis
    sepsis_risk = 0.05 * (age - 65) / 15 + 0.3 * comorbidity + np.random.normal(0, 0.2, n_patients)
    sepsis = (sepsis_risk > 0.2).astype(int)

    # PRE-SEPSIS measurements (legitimate features)
    hr_before = baseline_hr + sepsis * 10 + np.random.normal(0, 5, n_patients)  # Slight elevation
    wbc_before = baseline_wbc + sepsis * 2 + np.random.normal(0, 1, n_patients)

    # POST-SEPSIS measurements (LEAKAGE!)
    # After sepsis is diagnosed, patients get sicker AND more tests ordered
    hr_after = baseline_hr + sepsis * 30 + np.random.normal(0, 8, n_patients)  # Major elevation
    wbc_after = baseline_wbc + sepsis * 8 + np.random.normal(0, 2, n_patients)  # Leukocytosis
    lactate = 1.0 + sepsis * 3.5 + np.random.normal(0, 0.5, n_patients)  # Often ordered ONLY if sepsis suspected

    df = pd.DataFrame({
        'patient_id': [f'P{i:04d}' for i in range(n_patients)],
        'age': age,
        'comorbidity': comorbidity,
        'hr_before': hr_before,  # Legitimate
        'wbc_before': wbc_before,  # Legitimate
        'hr_after': hr_after,  # LEAKAGE
        'wbc_after': wbc_after,  # LEAKAGE
        'lactate': lactate,  # LEAKAGE (only ordered if sepsis suspected)
        'sepsis': sepsis
    })

    return df

df_sepsis = generate_sepsis_data_with_leakage(1000)

print("Sepsis dataset generated:")
print(f"Patients: {len(df_sepsis)}")
print(f"Sepsis prevalence: {df_sepsis['sepsis'].mean():.1%}")
print("\nFeatures:")
print("  Legitimate: age, comorbidity, hr_before, wbc_before")
print("  LEAKAGE: hr_after, wbc_after, lactate")
df_sepsis.head(10)

In [ ]:
# Compare models WITH and WITHOUT leakage
def evaluate_with_and_without_leakage(df):
    """
    Train models with legitimate features vs leaked features.
    """
    # Legitimate features only
    X_legit = df[['age', 'comorbidity', 'hr_before', 'wbc_before']]

    # WITH LEAKAGE (includes post-sepsis measurements)
    X_leaked = df[['age', 'comorbidity', 'hr_before', 'wbc_before',
                   'hr_after', 'wbc_after', 'lactate']]

    y = df['sepsis']

    # Train-test split
    X_legit_train, X_legit_test, y_train, y_test = train_test_split(
        X_legit, y, test_size=0.3, random_state=42
    )
    X_leaked_train, X_leaked_test, _, _ = train_test_split(
        X_leaked, y, test_size=0.3, random_state=42
    )

    # Model 1: Legitimate features
    clf_legit = RandomForestClassifier(n_estimators=100, random_state=42)
    clf_legit.fit(X_legit_train, y_train)
    y_pred_legit = clf_legit.predict_proba(X_legit_test)[:, 1]
    auroc_legit = roc_auc_score(y_test, y_pred_legit)

    # Model 2: WITH LEAKAGE
    clf_leaked = RandomForestClassifier(n_estimators=100, random_state=42)
    clf_leaked.fit(X_leaked_train, y_train)
    y_pred_leaked = clf_leaked.predict_proba(X_leaked_test)[:, 1]
    auroc_leaked = roc_auc_score(y_test, y_pred_leaked)

    print("="*70)
    print("TARGET LEAKAGE DEMONSTRATION")
    print("="*70)
    print(f"\nModel WITHOUT leakage (legitimate features): AUROC = {auroc_legit:.3f}")
    print(f"Model WITH leakage (includes post-sepsis data): AUROC = {auroc_leaked:.3f}")
    print(f"\n💥 Artificial boost from leakage: {(auroc_leaked - auroc_legit):.3f}")
    print("\n⚠️ Model WITH leakage looks amazing but WILL FAIL in deployment!")
    print("    (Post-sepsis data won't be available when making real-time predictions)")
    print("="*70)

    # Feature importance
    importance_leaked = pd.DataFrame({
        'feature': X_leaked.columns,
        'importance': clf_leaked.feature_importances_
    }).sort_values('importance', ascending=False)

    print("\nFeature Importance (Leaked Model):")
    print(importance_leaked)
    print("\n💡 Leaked features dominate! Model relies on unavailable information.")

    return auroc_legit, auroc_leaked, y_test, y_pred_legit, y_pred_leaked

auroc_legit, auroc_leaked, y_test, y_pred_legit, y_pred_leaked = evaluate_with_and_without_leakage(df_sepsis)

In [ ]:
# Visualize ROC curves
fig, ax = plt.subplots(figsize=(10, 7))

# ROC curve without leakage
fpr_legit, tpr_legit, _ = roc_curve(y_test, y_pred_legit)
ax.plot(fpr_legit, tpr_legit, linewidth=3, label=f'Legitimate Features (AUROC={auroc_legit:.3f})', color='steelblue')

# ROC curve with leakage
fpr_leaked, tpr_leaked, _ = roc_curve(y_test, y_pred_leaked)
ax.plot(fpr_leaked, tpr_leaked, linewidth=3, label=f'WITH LEAKAGE (AUROC={auroc_leaked:.3f})', color='red', linestyle='--')

# Diagonal reference
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random (AUROC=0.5)')

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('Impact of Target Leakage on ROC Curve', fontweight='bold', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 The leaked model appears 'too good to be true' - and it is!")

## 2. Type 2: Temporal Leakage

**Definition**: Using future information to predict past events.

**Example**: Using tomorrow's blood pressure to predict today's mortality risk

In [ ]:
def generate_time_series_with_temporal_leakage(n_patients=200, hours=48):
    """
    Generate time series ICU data.
    Goal: Predict mortality at hour 24 using hours 0-23 only.
    """
    data = []

    for patient_id in range(n_patients):
        # Patient risk level
        risk = np.random.rand()

        # Mortality at hour 24
        mortality = (risk > 0.7)

        for hour in range(hours):
            # Vitals deteriorate if high risk
            hr = 70 + risk * 20 + hour * risk * 0.5 + np.random.normal(0, 5)
            sbp = 120 - risk * 30 - hour * risk * 0.3 + np.random.normal(0, 10)

            # After hour 24 (mortality event), vitals get MUCH worse or flatline
            if hour >= 24 and mortality == 1:
                hr += 20 + (hour - 24) * 2
                sbp -= 20 + (hour - 24) * 1

            data.append({
                'patient_id': patient_id,
                'hour': hour,
                'heart_rate': hr,
                'sbp': sbp,
                'mortality_24h': mortality
            })

    return pd.DataFrame(data)

df_temporal = generate_time_series_with_temporal_leakage(200, hours=48)

print("Time series dataset:")
print(f"Total records: {len(df_temporal)}")
print(f"Patients: {df_temporal['patient_id'].nunique()}")
print(f"Hours per patient: {df_temporal.groupby('patient_id').size().mean():.0f}")
print(f"Mortality rate: {df_temporal.groupby('patient_id')['mortality_24h'].first().mean():.1%}")
print("\nGoal: Predict mortality at hour 24 using ONLY hours 0-23")
df_temporal.head(15)

In [ ]:
def evaluate_temporal_leakage(df):
    """
    Compare prediction using correct time window vs leaked future data.
    """
    # Aggregate features for each patient
    # CORRECT: Use only hours 0-23 to predict mortality at hour 24
    df_correct = df[df['hour'] < 24].groupby('patient_id').agg({
        'heart_rate': ['mean', 'max'],
        'sbp': ['mean', 'min'],
        'mortality_24h': 'first'
    }).reset_index()
    df_correct.columns = ['patient_id', 'hr_mean', 'hr_max', 'sbp_mean', 'sbp_min', 'mortality']

    # WRONG WITH TEMPORAL LEAKAGE: Use hours 0-47 (includes FUTURE data after outcome!)
    df_leaked = df.groupby('patient_id').agg({
        'heart_rate': ['mean', 'max'],
        'sbp': ['mean', 'min'],
        'mortality_24h': 'first'
    }).reset_index()
    df_leaked.columns = ['patient_id', 'hr_mean', 'hr_max', 'sbp_mean', 'sbp_min', 'mortality']

    # Train models
    X_correct = df_correct[['hr_mean', 'hr_max', 'sbp_mean', 'sbp_min']]
    X_leaked = df_leaked[['hr_mean', 'hr_max', 'sbp_mean', 'sbp_min']]
    y = df_correct['mortality']

    X_correct_train, X_correct_test, y_train, y_test = train_test_split(
        X_correct, y, test_size=0.3, random_state=42
    )
    X_leaked_train, X_leaked_test, _, _ = train_test_split(
        X_leaked, y, test_size=0.3, random_state=42
    )

    # Model without temporal leakage
    clf_correct = RandomForestClassifier(n_estimators=100, random_state=42)
    clf_correct.fit(X_correct_train, y_train)
    y_pred_correct = clf_correct.predict_proba(X_correct_test)[:, 1]
    auroc_correct = roc_auc_score(y_test, y_pred_correct)

    # Model WITH temporal leakage
    clf_leaked = RandomForestClassifier(n_estimators=100, random_state=42)
    clf_leaked.fit(X_leaked_train, y_train)
    y_pred_leaked = clf_leaked.predict_proba(X_leaked_test)[:, 1]
    auroc_leaked = roc_auc_score(y_test, y_pred_leaked)

    print("="*70)
    print("TEMPORAL LEAKAGE DEMONSTRATION")
    print("="*70)
    print(f"\nModel using hours 0-23 ONLY (correct): AUROC = {auroc_correct:.3f}")
    print(f"Model using hours 0-47 (WITH LEAKAGE): AUROC = {auroc_leaked:.3f}")
    print(f"\n💥 Artificial boost from temporal leakage: {(auroc_leaked - auroc_correct):.3f}")
    print("\n⚠️ At prediction time (hour 24), future data (hours 25-47) doesn't exist!")
    print("    Model will fail catastrophically in deployment.")
    print("="*70)

evaluate_temporal_leakage(df_temporal)

## 3. Type 3: Train-Test Leakage (Preprocessing)

**Definition**: Fitting preprocessing on full dataset before splitting.

**Example**: StandardScaler fit on train+test, then split

In [ ]:
# Generate simple dataset
X = np.random.normal(100, 20, (1000, 5))
y = (X[:, 0] + X[:, 1] > 200).astype(int)

# WRONG: Scale before splitting
scaler_wrong = StandardScaler()
X_scaled_wrong = scaler_wrong.fit_transform(X)
X_train_wrong, X_test_wrong, y_train, y_test = train_test_split(
    X_scaled_wrong, y, test_size=0.3, random_state=42
)

clf_wrong = RandomForestClassifier(n_estimators=100, random_state=42)
clf_wrong.fit(X_train_wrong, y_train)
y_pred_wrong = clf_wrong.predict_proba(X_test_wrong)[:, 1]
auroc_wrong = roc_auc_score(y_test, y_pred_wrong)

# CORRECT: Split first, then scale
X_train_correct, X_test_correct, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

scaler_correct = StandardScaler()
X_train_correct = scaler_correct.fit_transform(X_train_correct)  # Fit on training only
X_test_correct = scaler_correct.transform(X_test_correct)  # Transform test using training statistics

clf_correct = RandomForestClassifier(n_estimators=100, random_state=42)
clf_correct.fit(X_train_correct, y_train)
y_pred_correct = clf_correct.predict_proba(X_test_correct)[:, 1]
auroc_correct = roc_auc_score(y_test, y_pred_correct)

print("="*70)
print("TRAIN-TEST LEAKAGE (Preprocessing)")
print("="*70)
print(f"\nWRONG (scale before split): AUROC = {auroc_wrong:.3f}")
print(f"CORRECT (split then scale): AUROC = {auroc_correct:.3f}")
print(f"\n💥 Optimism bias from preprocessing leakage: {(auroc_wrong - auroc_correct):.3f}")
print("\n⚠️ Test set statistics leaked into training preprocessing!")
print("="*70)

## 4. Proper Preprocessing Pipelines

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Proper way: Use sklearn Pipeline
# Pipeline ensures preprocessing is fit ONLY on training data in each CV fold

X_raw = np.random.normal(100, 20, (1000, 5))
# Introduce some missing values
X_raw[np.random.rand(1000, 5) < 0.1] = np.nan
y = (np.nansum(X_raw[:, :2], axis=1) > 200).astype(int)

# Split data FIRST
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.3, random_state=42
)

# Create pipeline: imputation → scaling → model
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Fit pipeline on training data only
pipeline.fit(X_train, y_train)

# Evaluate on test data
# Pipeline automatically applies same preprocessing (fit on train) to test
y_pred = pipeline.predict_proba(X_test)[:, 1]
auroc_pipeline = roc_auc_score(y_test, y_pred)

print("="*70)
print("PROPER PIPELINE APPROACH")
print("="*70)
print("\nPipeline steps:")
print("  1. Split data (train/test)")
print("  2. Fit imputer on training data")
print("  3. Fit scaler on training data")
print("  4. Fit classifier on training data")
print("  5. Apply fitted transformations to test data")
print(f"\nResult: AUROC = {auroc_pipeline:.3f}")
print("\n✓ No leakage! Test data never seen during preprocessing fitting.")
print("="*70)

## 5. Leakage Detection Checklist

In [ ]:
print("="*70)
print("DATA LEAKAGE DETECTION CHECKLIST")
print("="*70)

print("\n🔍 TARGET LEAKAGE")
print("  □ Are any features collected AFTER the outcome event?")
print("  □ Do features contain information caused BY the outcome?")
print("  □ Are there features that would be unavailable at prediction time?")
print("  □ Check: Does performance seem 'too good to be true' (AUROC > 0.95)?")

print("\n⏰ TEMPORAL LEAKAGE")
print("  □ Is data timestamped correctly?")
print("  □ Are features using ONLY past/present data (not future)?")
print("  □ For time series: Is train data strictly before test data?")
print("  □ Check: Did you sort by time before splitting?")

print("\n🔬 PREPROCESSING LEAKAGE")
print("  □ Did you split BEFORE preprocessing?")
print("  □ Is scaling/imputation fit ONLY on training data?")
print("  □ Is feature selection performed within CV folds?")
print("  □ Are you using sklearn Pipeline for preprocessing?")

print("\n👥 PATIENT-LEVEL LEAKAGE")
print("  □ For multiple records per patient: using GroupKFold?")
print("  □ Is same patient never in both train and test?")
print("  □ For multi-site data: testing on held-out sites?")

print("\n💊 DOMAIN-SPECIFIC CHECKS (Healthcare)")
print("  □ Lab ordered only AFTER suspicion of disease? (selection bias)")
print("  □ Treatment information included as feature? (confounding)")
print("  □ Using discharge diagnosis to predict admission risk? (impossible)")
print("  □ Using length of stay to predict in-hospital mortality? (correlated)")

print("\n" + "="*70)

## 6. Real-World Example: Detecting Leaked Features

In [ ]:
def detect_suspicious_features(X, y, threshold_auroc=0.90):
    """
    Detect features that individually achieve suspiciously high performance.
    Such features may contain leakage.
    """
    print("="*70)
    print("LEAKED FEATURE DETECTION")
    print("="*70)
    print(f"\nTesting each feature individually...")
    print(f"Flagging features with AUROC > {threshold_auroc} (suspicious!)\n")

    results = []

    for i, col in enumerate(X.columns):
        X_single = X[[col]]
        X_train, X_test, y_train, y_test = train_test_split(
            X_single, y, test_size=0.3, random_state=42
        )

        clf = RandomForestClassifier(n_estimators=50, random_state=42, max_depth=3)
        clf.fit(X_train, y_train)
        y_pred = clf.predict_proba(X_test)[:, 1]
        auroc = roc_auc_score(y_test, y_pred)

        flag = "🚩 SUSPICIOUS!" if auroc > threshold_auroc else ""
        results.append({'feature': col, 'auroc': auroc, 'flag': flag})

        print(f"  {col:20s}: AUROC = {auroc:.3f} {flag}")

    print("\n" + "="*70)
    print("💡 Suspicious features likely contain leakage or are proxies for outcome!")
    print("="*70)

    return pd.DataFrame(results).sort_values('auroc', ascending=False)

# Test on our sepsis data
X_test_leakage = df_sepsis[['age', 'comorbidity', 'hr_before', 'wbc_before',
                             'hr_after', 'wbc_after', 'lactate']]
y_test_leakage = df_sepsis['sepsis']

results = detect_suspicious_features(X_test_leakage, y_test_leakage, threshold_auroc=0.75)

## 7. Key Takeaways

### What We Learned

1. **Target Leakage**:
   - Using features created by or after the outcome
   - Example: Post-sepsis labs to predict sepsis onset
   - Detection: Suspiciously high performance (AUROC > 0.95)

2. **Temporal Leakage**:
   - Using future information to predict past events
   - Example: Tomorrow's vitals to predict today's mortality
   - Prevention: Strict temporal ordering, careful feature construction

3. **Preprocessing Leakage**:
   - Fitting transformations on full dataset before splitting
   - Example: StandardScaler fit on train+test
   - Solution: Always split first, use sklearn Pipeline

4. **Impact of Leakage**:
   - Artificial AUROC boost: 0.05-0.20+
   - Models appear to work but fail completely in production
   - Wasted development time and resources

### Clinical Considerations

- **Documentation timing**: Labs ordered AFTER clinical suspicion
- **Treatment as feature**: Medications given after diagnosis
- **Discharge diagnoses**: Not available at admission
- **Length of stay**: Correlated with severity but unavailable prospectively
- **Selection bias**: Missing labs often mean patient not sick enough to warrant test

### Best Practices

1. **Always use sklearn Pipeline** for preprocessing
2. **Split data BEFORE any preprocessing**
3. **Timestamp all data** and verify temporal ordering
4. **Review features with domain experts** for temporal feasibility
5. **Test each feature individually** for suspicious performance
6. **Document data collection timing** explicitly
7. **Use proper CV strategies** (GroupKFold, TimeSeriesSplit)
8. **Be skeptical of perfect performance** - investigate thoroughly

### Red Flags

- AUROC > 0.95 on complex clinical outcome
- Perfect or near-perfect predictions
- Single feature achieves AUROC > 0.90
- Model performs much worse in deployment than development
- Features that "shouldn't" be predictive dominate importance

### Connection to Journey Stories

**Marcus (Sepsis)**: Must use only pre-sepsis onset data. Labs ordered after sepsis recognition would leak target (see Notebook 7.3)

**Yuki (AFib)**: Can't use cardiologist's diagnosis timestamp as feature - that's the target! Must use only wearable data before diagnosis

**All Journeys**: Leakage is why "amazing" development results often fail in practice. Rigorous temporal validation essential.

---

*This notebook is part of "AI in Healthcare" (Volume 1, Chapter 4: Data Preparation)*

## Exercises

1. **Leakage Hunt**: Review a public healthcare ML paper. Identify potential sources of leakage in their methodology.

2. **Temporal Validation**: Implement a proper temporal validation scheme for predicting ICU mortality 24 hours in advance.

3. **Feature Audit**: Create a systematic feature audit tool that checks each feature for temporal feasibility given outcome timing.

4. **Pipeline Practice**: Build a complete Pipeline including imputation, scaling, feature selection, and modeling. Verify no leakage.

5. **Cross-Validation Leakage**: Deliberately introduce feature selection outside CV folds. Quantify the optimism bias created.